In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [2]:
# =========================
# Configuration
# =========================

root = os.getcwd()
root

'/home/peiretti/fair-clustering/results/synthetic/clus3_rc'

In [3]:
algorithm = "taucc_fair_max"
CSV_PATH = root + f"/{algorithm}/aggregated_groups2.csv"
CSV_PATH

'/home/peiretti/fair-clustering/results/synthetic/clus3_rc/taucc_fair_max/aggregated_groups2.csv'

In [4]:
OUTPUT_DIR = root + "/plot"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_DIR += f"/{algorithm}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
OUTPUT_DIR

'/home/peiretti/fair-clustering/results/synthetic/clus3_rc/plot/taucc_fair_max'

In [6]:
# Main metrics available in your CSV
TAU_X = "tau_x_mean"
TAU_Y = "tau_y_mean"
BALANCE = "balance_bera_mean"

# Derived metrics used in some plots
QUALITY_AGG_NAME = "quality_mean"

# Filtering tolerance for float comparisons
TOL = 1e-9

METRIC_LABELS = {
    TAU_X: r"tau_x",
    TAU_Y: r"tau_y",
    BALANCE: "Balance"
}


In [7]:
# =========================
# Utility functions
# =========================

def load_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # Add aggregated metrics that are often useful in plots
    df[QUALITY_AGG_NAME] = (df[TAU_X] + df[TAU_Y]) / 2.0
    return df


def close_to(series: pd.Series, value: float, tol: float = TOL) -> pd.Series:
    return np.isclose(series.astype(float), value, atol=tol)


def filter_equal_alpha(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only rows where:
    row_fair_majority == row_fair_minority
    col_fair_majority == col_fair_minority
    This corresponds to the simplified setting:
    alpha_rows = (a_r, a_r), alpha_cols = (a_c, a_c)
    """
    mask = (
        np.isclose(df["row_fair_majority"], df["row_fair_minority"], atol=TOL) &
        np.isclose(df["col_fair_majority"], df["col_fair_minority"], atol=TOL)
    )
    out = df.loc[mask].copy()
    out["alpha_rows"] = out["row_fair_majority"]
    out["alpha_cols"] = out["col_fair_majority"]
    return out


def make_pivot(df: pd.DataFrame, index_col: str, column_col: str, value_col: str) -> pd.DataFrame:
    """
    Create pivot table using mean in case of duplicate combinations.
    """
    pivot = pd.pivot_table(
        df,
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc="mean"
    )
    pivot = pivot.sort_index().sort_index(axis=1)
    return pivot


def plot_heatmap(
    pivot: pd.DataFrame,
    title: str,
    xlabel: str,
    ylabel: str,
    output_path: str,
    fmt: str = ".3f",
    cmap: str = "viridis",
    cbar_label: str = "",
    vmin=None,
    vmax=None
):
    
    fig, ax = plt.subplots(figsize=(8, 6))
    #im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap=cmap)
    im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)

    # ticks
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels([f"{x:.2f}" if isinstance(x, (float, np.floating)) else str(x) for x in pivot.columns], rotation=45, ha="right")
    ax.set_yticklabels([f"{y:.2f}" if isinstance(y, (float, np.floating)) else str(y) for y in pivot.index])

    # labels
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=16)

    # cell annotations
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, format(val, fmt), ha="center", va="center", fontsize=12)

    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.set_ylabel(cbar_label, rotation=270, labelpad=15, fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    #plt.show()
    plt.close(fig)


def plot_scatter_quality_fairness(
    df: pd.DataFrame,
    title: str,
    output_path: str,
    color_col: str = "sensitive_px",
    annotate: bool = False
):
    """
    Scatter plot:
    x = average tau
    y = balance
    color = chosen variable
    """
    fig, ax = plt.subplots(figsize=(7, 5))

    sc = ax.scatter(
        df[QUALITY_AGG_NAME],
        df[BALANCE],
        c=df[color_col],
        s=60,
        alpha=0.8
    )

    ax.set_xlabel(r"Average quality $(\tau_{R|C} + \tau_{C|R})/2$")
    ax.set_ylabel("Balance")
    ax.set_title(title)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.ax.set_ylabel(color_col, rotation=270, labelpad=15)

    if annotate:
        for _, row in df.iterrows():
            label = (
                f"ar=({row['row_fair_majority']:.1f},{row['row_fair_minority']:.1f})\n"
                f"ac=({row['col_fair_majority']:.1f},{row['col_fair_minority']:.1f})"
            )
            ax.annotate(label, (row[QUALITY_AGG_NAME], row[BALANCE]), fontsize=7, alpha=0.7)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    #plt.show()
    plt.close(fig)


def plot_line_slice(
    df,
    x_col,
    y_cols,
    title,
    output_path
):
    """
    Simple line plot for multiple metrics over one x variable.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    df_sorted = df.sort_values(x_col)

    for y_col in y_cols:
        ax.plot(df_sorted[x_col], df_sorted[y_col], marker="o", label=y_col)

    ax.set_xlabel(x_col)
    ax.set_ylabel("Metric value")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    #plt.show()
    plt.close(fig)



In [10]:
# =========================
# Plot 1:
# Heatmap alpha_rows vs alpha_cols
# under symmetric fairness vectors
# =========================

def heatmap_alpha_rows_vs_alpha_cols(
    df: pd.DataFrame,
    sensitive_px: float,
    sensitive_py: float,
    metric: str,
    output_name: str
):
    dfe = filter_equal_alpha(df)

    dff = dfe[
        close_to(dfe["sensitive_px"], sensitive_px) &
        close_to(dfe["sensitive_py"], sensitive_py)
    ].copy()

    pivot = make_pivot(dff, "alpha_rows", "alpha_cols", metric)
    
    vmin, vmax = (0, 1) if metric == BALANCE else (None, None)

    plot_heatmap(
        pivot=pivot,
        title=(
            f"{METRIC_LABELS.get(metric, metric)}\n"
            f"(sensitive_dx={sensitive_px:.2f}, sensitive_dy={sensitive_py:.2f})"
        ),
        xlabel=r"$\alpha$ columns",
        ylabel=r"$\alpha$ rows",
        output_path=os.path.join(OUTPUT_DIR, output_name),
        cbar_label=METRIC_LABELS.get(metric, metric),
        vmin=vmin,
        vmax=vmax
    )

In [ ]:
# =========================
# Plot 2:
# Heatmap sensitive_px vs sensitive_py
# for fixed symmetric fairness vectors
# =========================

def heatmap_sensitive_rows_vs_sensitive_cols(
    df: pd.DataFrame,
    alpha_rows: float,
    alpha_cols: float,
    metric: str,
    output_name: str
):
    dfe = filter_equal_alpha(df)

    dff = dfe[
        close_to(dfe["alpha_rows"], alpha_rows) &
        close_to(dfe["alpha_cols"], alpha_cols)
    ].copy()

    pivot = make_pivot(dff, "sensitive_py", "sensitive_px", metric)
    
    vmin, vmax = (0, 1) if metric == BALANCE else (None, None)

    plot_heatmap(
        pivot=pivot,
        title=(
            f"{METRIC_LABELS.get(metric, metric)} across row/column sensitive degrees\n"
            f"(alpha_rows={alpha_rows:.2f}, alpha_cols={alpha_cols:.2f})"
        ),
        xlabel="sensitive_degree (rows)",
        ylabel="sensitive_degree (cols)",
        output_path=os.path.join(OUTPUT_DIR, output_name),
        cbar_label=METRIC_LABELS.get(metric, metric),
        vmin=vmin,
        vmax=vmax
    )

In [ ]:
# =========================
# Plot 3:
# Heatmap row alpha majority vs minority
# with columns fixed
# =========================

def heatmap_row_majority_vs_minority(
    df: pd.DataFrame,
    sensitive_px: float,
    sensitive_py: float,
    col_alpha_majority: float,
    col_alpha_minority: float,
    metric: str,
    output_name: str
):
    dff = df[
        close_to(df["sensitive_px"], sensitive_px) &
        close_to(df["sensitive_py"], sensitive_py) &
        close_to(df["col_fair_majority"], col_alpha_majority) &
        close_to(df["col_fair_minority"], col_alpha_minority)
    ].copy()

    pivot = make_pivot(dff, "row_fair_minority", "row_fair_majority", metric)
    
    vmin, vmax = (0, 1) if metric == BALANCE else (None, None)

    plot_heatmap(
        pivot=pivot,
        title=(
            f"{METRIC_LABELS.get(metric, metric)} for alpha_rows\n"
            f"(sensitive_px={sensitive_px:.2f}, sensitive_py={sensitive_py:.2f}, "
            f"alpha_cols=({col_alpha_majority:.2f},{col_alpha_minority:.2f}))"
        ),
        xlabel="alpha_rows (majority)",
        ylabel="alpha_rows (minority)",
        output_path=os.path.join(OUTPUT_DIR, output_name),
        cbar_label=METRIC_LABELS.get(metric, metric),
        vmin=vmin,
        vmax=vmax
    )


In [ ]:
# =========================
# Plot 4:
# Scatter quality vs fairness
# =========================

def scatter_quality_vs_fairness_fixed_sensitive(
    df: pd.DataFrame,
    sensitive_px: float,
    sensitive_py: float,
    output_name: str
):
    dff = df[
        close_to(df["sensitive_px"], sensitive_px) &
        close_to(df["sensitive_py"], sensitive_py)
    ].copy()

    plot_scatter_quality_fairness(
        df=dff,
        title=(
            f"Quality vs fairness trade-off\n"
            f"(sensitive_px={sensitive_px:.2f}, sensitive_py={sensitive_py:.2f})"
        ),
        output_path=os.path.join(OUTPUT_DIR, output_name),
        color_col="row_fair_majority",
        annotate=False
    )

In [ ]:
# =========================
# Plot 5:
# Line plot over sensitive degree
# with symmetric fairness vectors
# =========================

def lineplot_over_sensitive_px(
    df: pd.DataFrame,
    alpha_rows: float,
    alpha_cols: float,
    sensitive_py_fixed: float,
    output_name: str
):
    dfe = filter_equal_alpha(df)

    dff = dfe[
        close_to(dfe["alpha_rows"], alpha_rows) &
        close_to(dfe["alpha_cols"], alpha_cols) &
        close_to(dfe["sensitive_py"], sensitive_py_fixed)
    ].copy()

    plot_line_slice(
        df=dff,
        x_col="sensitive_px",
        y_cols=[TAU_X, TAU_Y, BALANCE],
        title=(
            f"Metrics vs sensitive_px\n"
            f"(alpha_rows={alpha_rows:.2f}, alpha_cols={alpha_cols:.2f}, "
            f"sensitive_py={sensitive_py_fixed:.2f})"
        ),
        output_path=os.path.join(OUTPUT_DIR, output_name)
    )


In [11]:
# =========================
# Main
# =========================

df = load_data(CSV_PATH)

sensitive_px=0.8
sensitive_py=0.8

# ---------------------------------
# Example 1:
# alpha_rows vs alpha_cols heatmaps
# in a hard scenario
# ---------------------------------
for metric in [TAU_X, TAU_Y, BALANCE]:
    heatmap_alpha_rows_vs_alpha_cols(
        df=df,
        sensitive_px=sensitive_px,
        sensitive_py=sensitive_py,
        metric=metric,
        output_name=f"heatmap_alphaRows_alphaCols_{metric}.png"
    )

"""
# ---------------------------------
# Example 2:
# sensitive degree heatmaps
# for different fairness levels
# ---------------------------------
for alpha_rows, alpha_cols in [(0.0, 0.0), (0.5, 0.5), (1.0, 1.0)]:
    for metric in [TAU_X, TAU_Y, BALANCE]:
        heatmap_sensitive_rows_vs_sensitive_cols(
            df=df,
            alpha_rows=alpha_rows,
            alpha_cols=alpha_cols,
            metric=metric,
            output_name=(
                f"heatmap_sensitiveRows_sensitiveCols_"
                f"ar{alpha_rows:.1f}_ac{alpha_cols:.1f}_{metric}.png"
            )
        )

# ---------------------------------
# Example 3:
# row fairness vector heatmap
# fixing column fairness vector
# ---------------------------------
for metric in [TAU_X, TAU_Y, BALANCE]:
    heatmap_row_majority_vs_minority(
        df=df,
        sensitive_px=sensitive_px,
        sensitive_py=sensitive_py,
        col_alpha_majority=1.0,
        col_alpha_minority=1.0,
        metric=metric,
        output_name=f"heatmap_rowMaj_rowMin_{metric}.png"
    )

# ---------------------------------
# Example 4:
# quality vs fairness scatter
# ---------------------------------
scatter_quality_vs_fairness_fixed_sensitive(
    df=df,
    sensitive_px=sensitive_px,
    sensitive_py=sensitive_py,
    output_name="scatter_quality_vs_fairness_sensitive_1_1.png"
)

# ---------------------------------
# Example 5:
# line plot over row sensitive degree
# ---------------------------------
lineplot_over_sensitive_px(
    df=df,
    alpha_rows=1.0,
    alpha_cols=1.0,
    sensitive_py_fixed=0.8,
    output_name="lineplot_sensitive_px_alpha_1_1.png"
)

print(f"Plots saved in: {OUTPUT_DIR}")
"""

'\n# ---------------------------------\n# Example 2:\n# sensitive degree heatmaps\n# for different fairness levels\n# ---------------------------------\nfor alpha_rows, alpha_cols in [(0.0, 0.0), (0.5, 0.5), (1.0, 1.0)]:\n    for metric in [TAU_X, TAU_Y, BALANCE]:\n        heatmap_sensitive_rows_vs_sensitive_cols(\n            df=df,\n            alpha_rows=alpha_rows,\n            alpha_cols=alpha_cols,\n            metric=metric,\n            output_name=(\n                f"heatmap_sensitiveRows_sensitiveCols_"\n                f"ar{alpha_rows:.1f}_ac{alpha_cols:.1f}_{metric}.png"\n            )\n        )\n\n# ---------------------------------\n# Example 3:\n# row fairness vector heatmap\n# fixing column fairness vector\n# ---------------------------------\nfor metric in [TAU_X, TAU_Y, BALANCE]:\n    heatmap_row_majority_vs_minority(\n        df=df,\n        sensitive_px=sensitive_px,\n        sensitive_py=sensitive_py,\n        col_alpha_majority=1.0,\n        col_alpha_minority=1

In [ ]:
OUTPUT_DIR